# Two-Part Genomic Selection Tutorial - Hybrid Breeding

This notebook replicates the AlphaSimR two-part genomic selection hybrid breeding tutorial using AlphaSimPy.
It demonstrates a two-part breeding strategy with rapid cycling of parents in population improvement
and conventional breeding for product development. Applies GS to advance individuals from DH to YT1
as well as in population improvement.

**Authors**: Translated from AlphaSimR tutorial by Jon Bancic, Philip Greenspoon, Chris Gaynor, Gregor Gorjanc  
**Date**: 2024  
**Package**: AlphaSimPy

**Strategy**: Uses two-part strategy with rapid cycling of parents in population improvement
and conventional breeding for product development. Applies GS to advance individuals from DH to YT1
as well as in population improvement.

## Import Required Libraries

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import solve
from AlphaSimPy import (
    runMacs, SimParam, newPop, randCross, setPheno, selectInd,
    meanG, varG, makeDH, hybridCross, setPhenoGCA, calcGCA, mergePops,
    selectWithinFam
)

print("AlphaSimPy Hybrid Breeding - Two-Part Genomic Selection Tutorial")
print("All libraries imported successfully!")

AlphaSimPy Hybrid Breeding - Two-Part Genomic Selection Tutorial
All libraries imported successfully!


In [1]:
1

1

## Helper Functions for Genomic Selection

These functions implement RRBLUP and setEBV functionality for genomic selection, plus a helper for subsetting Pop objects.

In [9]:
def subsetPop(pop, indices):
    """
    Subset a Pop object by indices (helper function since Pop doesn't support indexing).
    
    Parameters:
    -----------
    pop : Pop
        Population object
    indices : list or slice
        Indices to select
    
    Returns:
    --------
    Pop
        Subsetted population
    """
    from AlphaSimPy import Pop
    
    if isinstance(indices, slice):
        indices = list(range(*indices.indices(pop.n_ind)))
    
    if not indices:
        # Return empty population
        return Pop(
            n_ind=0, n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
            geno=[], gen_map=pop.gen_map, centromere=pop.centromere,
            inbred=pop.inbred, id=[], iid=[], mother=[], father=[],
            sex=[], n_traits=pop.n_traits, gv=np.empty((0, pop.n_traits)),
            pheno=np.empty((0, pop.n_traits)), ebv=np.empty((0, 0)),
            gxe=pop.gxe, fix_eff=[], misc={}, misc_pop={}
        )
    
    return Pop(
        n_ind=len(indices), n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
        geno=[pop.geno[chr_idx][:, :, indices] for chr_idx in range(pop.n_chr)],
        gen_map=pop.gen_map, centromere=pop.centromere, inbred=pop.inbred,
        id=[pop.id[i] for i in indices],
        iid=[pop.iid[i] for i in indices],
        mother=[pop.mother[i] for i in indices],
        father=[pop.father[i] for i in indices],
        sex=[pop.sex[i] for i in indices],
        n_traits=pop.n_traits, gv=pop.gv[indices, :],
        pheno=pop.pheno[indices, :], ebv=pop.ebv[indices, :],
        gxe=pop.gxe, fix_eff=[pop.fix_eff[i] for i in indices],
        misc=pop.misc, misc_pop=pop.misc_pop
    )


def pullSnpGeno(pop, simParam, snpChip=1):
    """
    Extract SNP genotype matrix from a population.
    
    Parameters:
    -----------
    pop : Pop
        Population object
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use (1-indexed)
    
    Returns:
    --------
    np.ndarray
        Matrix of SNP genotypes (n_ind x n_snp)
    """
    if snpChip < 1 or snpChip > len(simParam.snp_chips):
        raise ValueError(f"snpChip {snpChip} not available")
    
    snp_chip = simParam.snp_chips[snpChip - 1]
    n_snp = sum(snp_chip.loci_per_chr)
    
    if n_snp == 0:
        raise ValueError("No SNPs available in specified chip")
    
    # Extract genotypes for SNP loci
    geno_matrix = np.zeros((pop.n_ind, n_snp), dtype=np.float64)
    
    snp_idx = 0
    for chr_idx in range(pop.n_chr):
        n_snp_chr = snp_chip.loci_per_chr[chr_idx]
        if n_snp_chr == 0:
            continue
        
        # Get SNP locations on this chromosome
        snp_loci = snp_chip.loci_loc[snp_idx:snp_idx + n_snp_chr]
        
        # Extract genotypes from packed format
        for ind_idx in range(pop.n_ind):
            for snp_loc_idx, snp_loc in enumerate(snp_loci):
                # Extract genotype from packed byte format
                byte_idx = snp_loc // 8
                bit_idx = snp_loc % 8
                
                # Sum across ploidy
                genotype = 0
                for p in range(pop.ploidy):
                    byte_val = pop.geno[chr_idx][byte_idx, p, ind_idx]
                    bit_val = (byte_val >> bit_idx) & 1
                    genotype += bit_val
                
                geno_matrix[ind_idx, snp_idx + snp_loc_idx] = genotype
        
        snp_idx += n_snp_chr
    
    return geno_matrix


def RRBLUP(trainPop, simParam, traits=1, use="pheno", snpChip=1):
    """
    Fit an RR-BLUP model for genomic predictions.
    
    Parameters:
    -----------
    trainPop : Pop
        Training population
    simParam : SimParam
        Simulation parameters
    traits : int
        Trait to model (1-indexed)
    use : str
        Use "pheno", "gv", or "ebv" for training
    snpChip : int
        Which SNP chip to use
    
    Returns:
    --------
    dict
        Dictionary containing model coefficients and metadata
    """
    # Get response variable
    if use == "pheno":
        y = trainPop.pheno[:, traits - 1]
    elif use == "gv":
        y = trainPop.gv[:, traits - 1]
    elif use == "ebv":
        y = trainPop.ebv[:, traits - 1]
    else:
        raise ValueError(f"use='{use}' is not a valid option")
    
    # Remove missing values
    valid_idx = ~np.isnan(y)
    y = y[valid_idx]
    
    if len(y) == 0:
        raise ValueError("No valid observations for training")
    
    # Get SNP genotypes
    M = pullSnpGeno(trainPop, simParam, snpChip)
    M = M[valid_idx, :]
    
    # Center genotypes
    M_mean = np.mean(M, axis=0)
    M_centered = M - M_mean
    
    # Fit RR-BLUP using mixed model equations
    # y = Xb + Zu + e
    # where Z is the centered marker matrix
    # We use the GBLUP equivalent: K = ZZ'/p where p is number of markers
    
    n_markers = M_centered.shape[1]
    if n_markers == 0:
        raise ValueError("No markers available")
    
    # Calculate genomic relationship matrix G = ZZ' / p
    G = np.dot(M_centered, M_centered.T) / n_markers
    
    # Add small value to diagonal for numerical stability
    G += np.eye(G.shape[0]) * 1e-6
    
    # Estimate variance components (simplified - using fixed lambda)
    # In practice, you would estimate these, but for tutorial we use fixed values
    lambda_val = n_markers / 10.0  # Simplified lambda
    
    # Solve for BLUP: (G + lambda*I) * u = y
    # where u are the breeding values
    A = G + lambda_val * np.eye(G.shape[0])
    u = solve(A, y, assume_a='pos')
    
    # Store model for prediction
    model = {
        'u': u,  # BLUP solutions
        'M_mean': M_mean,  # Mean marker values for centering
        'M_train': M_centered,  # Training marker matrix (centered)
        'y_train': y,  # Training phenotypes
        'lambda': lambda_val,  # Regularization parameter
        'n_markers': n_markers,
        'trait': traits,
        'valid_idx': valid_idx
    }
    
    return model


def setEBV(pop, gsModel, simParam, snpChip=1):
    """
    Set estimated breeding values (EBV) for a population using a genomic selection model.
    
    Parameters:
    -----------
    pop : Pop
        Population to predict
    gsModel : dict
        Genomic selection model from RRBLUP
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use
    
    Returns:
    --------
    Pop
        Population with EBV set
    """
    # Get SNP genotypes for prediction population
    M_pred = pullSnpGeno(pop, simParam, snpChip)
    
    # Center using training population means
    M_pred_centered = M_pred - gsModel['M_mean']
    
    # Calculate genomic relationship between training and prediction
    # G_pred_train = M_pred_centered @ M_train' / p
    n_markers = gsModel['n_markers']
    G_pred_train = np.dot(M_pred_centered, gsModel['M_train'].T) / n_markers
    
    # Predict EBV: u_pred = G_pred_train @ (G_train + lambda*I)^(-1) @ y
    # We already have u_train = (G_train + lambda*I)^(-1) @ y from RRBLUP
    # So: u_pred = G_pred_train @ u_train
    
    # Calculate G_train for solving
    G_train = np.dot(gsModel['M_train'], gsModel['M_train'].T) / n_markers
    G_train += np.eye(G_train.shape[0]) * 1e-6
    
    # Solve: (G_train + lambda*I) * u = y
    A_train = G_train + gsModel['lambda'] * np.eye(G_train.shape[0])
    u_train = solve(A_train, gsModel['y_train'], assume_a='pos')
    
    # Predict: u_pred = G_pred_train @ u_train
    u_pred = np.dot(G_pred_train, u_train)
    
    # Set EBV in population
    if pop.ebv.shape[1] == 0:
        # Initialize EBV matrix if empty
        pop.ebv = np.zeros((pop.n_ind, 1))
    
    # Ensure EBV matrix has enough columns
    trait_idx = gsModel['trait'] - 1
    while pop.ebv.shape[1] <= trait_idx:
        pop.ebv = np.hstack([pop.ebv, np.zeros((pop.n_ind, 1))])
    
    pop.ebv[:, trait_idx] = u_pred
    
    return pop

## Global Parameters

Set up the simulation parameters for the two-part genomic selection hybrid breeding program.

In [10]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 20  # Number of years in burnin phase
n_future = 20  # Number of years in future phase
n_cycles = n_burnin + n_future
start_tp = 16  # Year to start training population

# Genome simulation
n_chr = 15  # Number of chromosomes
n_qtl = 300  # Number of QTL per chromosome
n_snp = 400  # Number of SNP per chromosome
n_gen_split = 100  # Heterotic pool split

# Initial inbred parents mean and variance
init_mean_g = 70  # bushels per acre
init_var_g = 20  # bushels per acre
# Degree of dominance
mean_dd = 0.92  # mean
var_dd = 0.3  # variance
# Error variances
init_var_ge = 40  # Genotype-by-year interaction
var_e = 270  # Yield trial error variance, bushels per acre
            # Relates to error variance for an entry mean

# Breeding program details
n_parents = 50  # Number of parents to start a breeding cycle
n_crosses = 80  # Number of crosses per year
fam_max = 15  # The maximum number of DH lines per cross
n_dh = 50  # DH lines produced per cross

# Effective replication of yield trials
rep_yt1 = 1  # h2 = 0.06
rep_yt2 = 2  # h2 = 0.11
rep_yt3 = 4  # h2 = 0.20
rep_yt4 = 8  # h2 = 0.34
rep_yt5 = 100  # h2 = 0.86

# Selection on GCA
# Number of inbreds per heterotic pool per stage
n_inbred1 = n_crosses * n_dh  # Do not change
n_inbred2 = 400
n_inbred3 = 40

# Number of testers per heterotic pool per stage
# Values must be smaller than n_elite
n_tester1 = 1
n_tester2 = 3

# Yield trial entries
n_yt1 = n_inbred1 * n_tester1  # Do not change
n_yt2 = n_inbred2 * n_tester2  # Do not change

# Selection on SCA

# Elite parents per heterotic pool
n_elite = 5

# Elite YT size
n_yt3 = n_inbred3 * n_elite  # Do not change
n_yt4 = 20
n_yt5 = 4

# Parameters for population improvement (two-part strategy)
n_cycles_pi = 2  # Number of cycles per year
n_parents_pi = 30  # Number of selected individuals per cycle
n_cross_male_pi = 100  # Number of male crosses per cycle
n_cross_female_pi = 100  # Number of female crosses per cycle
n_progeny_pi = 10  # Number of progeny per cross
max_fam_pi = 1  # Maximum number of selected individuals per cross
n_male_f1_pi = 80  # Number of F1-PI to advance to PD
n_female_f1_pi = 80  # Number of F1-PI to advance to PD

scenario_name = "HybridGSTP"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Start training population: Year {start_tp}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents per pool: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  DH lines per cross: {n_dh}")
print(f"\nTwo-Part Strategy Parameters:")
print(f"  PI cycles per year: {n_cycles_pi}")
print(f"  Parents per PI cycle: {n_parents_pi}")
print(f"  Male crosses per PI cycle: {n_cross_male_pi}")
print(f"  Female crosses per PI cycle: {n_cross_female_pi}")
print(f"  F1-PI to advance to PD: {n_male_f1_pi} male, {n_female_f1_pi} female")

Simulation Parameters:
  Replicates: 1
  Burn-in years: 20
  Future years: 20
  Total cycles: 40
  Start training population: Year 16
  Chromosomes: 15
  QTL per chromosome: 300
  SNP per chromosome: 400
  Parents per pool: 50
  Crosses per year: 80
  DH lines per cross: 50

Two-Part Strategy Parameters:
  PI cycles per year: 2
  Parents per PI cycle: 30
  Male crosses per PI cycle: 100
  Female crosses per PI cycle: 100
  F1-PI to advance to PD: 80 male, 80 female


## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.
The founders are split into two heterotic pools (male and female) to simulate hybrid breeding.

In [12]:
print("Creating founders...")

# Create founder population
# Split parameter creates two heterotic pools separated by n_gen_split generations
founder_pop = runMacs(
    nInd=n_parents * 2,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    inbred=True,
    split=n_gen_split,
    species="MAIZE"
)

print(f"✓ Created founder population: {founder_pop.n_ind} individuals")

# Set simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chips")

# Add traits: trait represents yield
# Using addTraitADG for additive, dominance, and GxE effects
SP.addTraitADG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    meanDD=mean_dd,
    varDD=var_dd,
    varGxE=init_var_ge
)
print(f"✓ Added TraitADG: {SP.n_traits} traits")

# Set permanent yield trial error variance
SP.setVarE(varE=var_e)
print(f"✓ Set error variance: {var_e}")

# Split heterotic pools to form initial parents
FemaleParents = newPop(founder_pop[0:n_parents], sim_param=SP)
MaleParents = newPop(founder_pop[n_parents:(n_parents * 2)], sim_param=SP)

print(f"✓ Created female parents: {FemaleParents.n_ind} individuals")
print(f"✓ Created male parents: {MaleParents.n_ind} individuals")

# Set hybrid parents for later yield trials
MaleElite = selectInd(MaleParents, nInd=n_elite, use="gv", simParam=SP)
FemaleElite = selectInd(FemaleParents, nInd=n_elite, use="gv", simParam=SP)

# Reverse order to keep best parent in longer
MaleElite = selectInd(MaleElite, nInd=n_elite, use="gv", simParam=SP)
FemaleElite = selectInd(FemaleElite, nInd=n_elite, use="gv", simParam=SP)
# Reverse the order by selecting in reverse
MaleElite_ids = MaleElite.id[::-1]
FemaleElite_ids = FemaleElite.id[::-1]
MaleElite = selectInd(MaleElite, nInd=n_elite, parents=[MaleElite.id.index(id) for id in MaleElite_ids], simParam=SP)
FemaleElite = selectInd(FemaleElite, nInd=n_elite, parents=[FemaleElite.id.index(id) for id in FemaleElite_ids], simParam=SP)

# Set initial testers for YT1 and YT2
# Requires nTesters to be smaller than nElite
MaleTester1 = selectInd(MaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
FemaleTester1 = selectInd(FemaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
MaleTester2 = selectInd(MaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)
FemaleTester2 = selectInd(FemaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)

print(f"✓ Created elite parents: {n_elite} per pool")
print(f"✓ Created testers: {n_tester1} for YT1, {n_tester2} for YT2")

print(f"\nFounder population summary:")
print(f"  Mean genetic value (female): {meanG(FemaleParents)[0]:.3f}")
print(f"  Mean genetic value (male): {meanG(MaleParents)[0]:.3f}")
print(f"  Genetic variance (female): {varG(FemaleParents)[0]:.3f}")
print(f"  Genetic variance (male): {varG(MaleParents)[0]:.3f}")

Creating founders...
100 2E8 -t 5E-6 -r 4E-6 -I 2 50 50 -eN 0.03 1 -eN 0.05 2 -eN 0.10 4 -eN 0.15 6 -eN 0.20 8 -eN 0.25 10 -eN 0.30 12 -eN 0.35 14 -eN 0.40 16 -eN 0.45 18 -eN 0.50 20 -eN 2.00 40 -eN 3.00 60 -eN 4.00 80 -eN 5.00 100 -ej 0.250001 2 1 -s 
✓ Created founder population: 100 individuals


ValueError: Not enough available loci on chromosome 1

## Fill Breeding Pipeline

Set up the initial breeding pipeline with 6 stages representing different evaluation years.
The pipeline includes:
- Stage 1: F1 crosses
- Stage 2: Doubled haploid (DH) lines and YT1 (GCA evaluation)
- Stage 3: YT2 (GCA evaluation)
- Stage 4: YT3 (Hybrid evaluation)
- Stage 5: YT4 (Hybrid evaluation)
- Stage 6: YT5 (Hybrid evaluation)

**Note**: Year effects (p parameter) are not yet fully supported in AlphaSimPy's `setPheno` function.
The GxE variance is still included in the trait definition, which affects genetic values.

In [ ]:
print("Filling breeding pipeline...")

# Set initial yield trials with unique individuals
# Sample year effects
P = np.random.uniform(size=6)

# Breeding program
for cohort in range(1, 7):
    print(f"  FillPipeline year: {cohort} of 6")
    
    # Stage 1
    MaleF1 = randCross(MaleParents, nCrosses=n_crosses, simParam=SP)
    FemaleF1 = randCross(FemaleParents, nCrosses=n_crosses, simParam=SP)
    
    # Stage 2
    if cohort < 6:
        p = P[6 - cohort]
        
        MaleDH = makeDH(MaleF1, nDH=n_dh, simParam=SP)
        FemaleDH = makeDH(FemaleF1, nDH=n_dh, simParam=SP)
        
        MaleYT1 = setPhenoGCA(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
        FemaleYT1 = setPhenoGCA(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
    
    # Stage 3
    if cohort < 5:
        p = P[5 - cohort]
        
        MaleYT2 = selectInd(MaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        FemaleYT2 = selectInd(FemaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        
        MaleYT2 = setPhenoGCA(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
        FemaleYT2 = setPhenoGCA(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
    
    # Stage 4
    if cohort < 4:
        p = P[4 - cohort]
        
        MaleInbredYT3 = selectInd(MaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        FemaleInbredYT3 = selectInd(FemaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        
        MaleHybridYT3 = hybridCross(MaleInbredYT3, FemaleElite, returnHybridPop=True, simParam=SP)
        FemaleHybridYT3 = hybridCross(FemaleInbredYT3, MaleElite, returnHybridPop=True, simParam=SP)
        
        MaleHybridYT3 = setPheno(MaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
        FemaleHybridYT3 = setPheno(FemaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
    
    # Stage 5
    if cohort < 3:
        p = P[3 - cohort]
        
        MaleHybridYT4 = selectInd(MaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        FemaleHybridYT4 = selectInd(FemaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        
        MaleHybridYT4 = setPheno(MaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        FemaleHybridYT4 = setPheno(FemaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        
        # Extract inbred parents from hybrid YT4
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = selectInd(MaleInbredYT3, nInd=len(MaleInbredYT4_ids), 
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], simParam=SP)
        FemaleInbredYT4 = selectInd(FemaleInbredYT3, nInd=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], simParam=SP)
    
    # Stage 6
    if cohort < 2:
        p = P[2 - cohort]
        
        MaleHybridYT5 = selectInd(MaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        FemaleHybridYT5 = selectInd(FemaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        
        MaleHybridYT5 = setPheno(MaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        FemaleHybridYT5 = setPheno(FemaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        
        # Extract inbred parents from hybrid YT5
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = selectInd(MaleInbredYT4, nInd=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], simParam=SP)
        FemaleInbredYT5 = selectInd(FemaleInbredYT4, nInd=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], simParam=SP)

print("\nPipeline filled successfully!")

## Main Simulation Loop

Run the breeding program simulation with burn-in (phenotypic selection) and future (two-part genomic selection) phases.
The two-part strategy includes rapid cycling of parents in population improvement (PI) and conventional breeding for product development (PD).

In [ ]:
# Create list to store results from reps
results = []
results_acc_pi = []

for REP in range(1, n_reps + 1):
    print(f"\n{'='*60}")
    print(f"Working on REP: {REP}")
    print(f"{'='*60}")
    
    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'meanG_inbred': [0.0] * n_cycles,
        'varG_inbred': [0.0] * n_cycles,
        'meanG_hybrid': [0.0] * n_cycles,
        'varG_hybrid': [0.0] * n_cycles,
        'acc_sel': [0.0] * n_cycles,
        'cor': [0.0] * n_cycles
    }
    
    # Create a data frame to track selection accuracy in every PI cycle
    acc_pi = {'accPI': [0.0] * (n_future * n_cycles_pi)}
    
    # Initialize training populations
    HybTrainPop = None
    MaleTrainPop = None
    FemaleTrainPop = None
    
    # Simulate year effects
    P = np.random.uniform(size=n_cycles)
    
    # Burn-in phase: Phenotypic selection
    print("\n--> Working on Burn-in")
    for year in range(1, n_burnin + 1):
        print(f" Working on burnin year: {year}")
        
        # Update parents (pick new parents)
        # Replace 10 oldest inbred parents with 10 new inbreds from YT4 stage
        if year > 1:
            # Keep oldest 40, add 10 new from YT4
            MaleParents_new = selectInd(MaleInbredYT4, nInd=10, use="pheno", simParam=SP)
            MaleParents_old = selectInd(MaleParents, nInd=n_parents - 10, 
                                      parents=list(range(10, n_parents)), simParam=SP)
            MaleParents = mergePops([MaleParents_old, MaleParents_new])
            
            FemaleParents_new = selectInd(FemaleInbredYT4, nInd=10, use="pheno", simParam=SP)
            FemaleParents_old = selectInd(FemaleParents, nInd=n_parents - 10,
                                        parents=list(range(10, n_parents)), simParam=SP)
            FemaleParents = mergePops([FemaleParents_old, FemaleParents_new])
        
        # Update testers (pick new testers)
        # Replace oldest hybrid parent with parent of best hybrid from YT5
        if year > 1:
            # Find best male inbred from YT5
            best_male_idx = np.argmax(MaleHybridYT5.pheno[:, 0])
            best_male_inbred_id = MaleHybridYT5.mother[best_male_idx]
            best_male_inbred_idx = MaleInbredYT5.id.index(best_male_inbred_id)
            MaleElite_new = selectInd(MaleInbredYT5, nInd=1, parents=[best_male_inbred_idx], simParam=SP)
            MaleElite_old = selectInd(MaleElite, nInd=n_elite - 1, parents=list(range(1, n_elite)), simParam=SP)
            MaleElite = mergePops([MaleElite_old, MaleElite_new])
            
            # Find best female inbred from YT5
            best_female_idx = np.argmax(FemaleHybridYT5.pheno[:, 0])
            best_female_inbred_id = FemaleHybridYT5.mother[best_female_idx]
            best_female_inbred_idx = FemaleInbredYT5.id.index(best_female_inbred_id)
            FemaleElite_new = selectInd(FemaleInbredYT5, nInd=1, parents=[best_female_inbred_idx], simParam=SP)
            FemaleElite_old = selectInd(FemaleElite, nInd=n_elite - 1, parents=list(range(1, n_elite)), simParam=SP)
            FemaleElite = mergePops([FemaleElite_old, FemaleElite_new])
            
            # Update testers
            MaleTester1 = selectInd(MaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
            FemaleTester1 = selectInd(FemaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
            MaleTester2 = selectInd(MaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)
            FemaleTester2 = selectInd(FemaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)
        
        # Advance year (advances yield trials by a year)
        p = P[year - 1]
        
        # Stage 6
        MaleHybridYT5 = selectInd(MaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        FemaleHybridYT5 = selectInd(FemaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        
        MaleHybridYT5 = setPheno(MaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        FemaleHybridYT5 = setPheno(FemaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = selectInd(MaleInbredYT4, nInd=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], simParam=SP)
        FemaleInbredYT5 = selectInd(FemaleInbredYT4, nInd=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], simParam=SP)
        
        # Stage 5
        MaleHybridYT4 = selectInd(MaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        FemaleHybridYT4 = selectInd(FemaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        
        MaleHybridYT4 = setPheno(MaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        FemaleHybridYT4 = setPheno(FemaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = selectInd(MaleInbredYT3, nInd=len(MaleInbredYT4_ids),
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], simParam=SP)
        FemaleInbredYT4 = selectInd(FemaleInbredYT3, nInd=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], simParam=SP)
        
        # Stage 4
        MaleInbredYT3 = selectInd(MaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        FemaleInbredYT3 = selectInd(FemaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        
        MaleHybridYT3 = hybridCross(MaleInbredYT3, FemaleElite, returnHybridPop=True, simParam=SP)
        FemaleHybridYT3 = hybridCross(FemaleInbredYT3, MaleElite, returnHybridPop=True, simParam=SP)
        
        MaleHybridYT3 = setPheno(MaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
        FemaleHybridYT3 = setPheno(FemaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
        
        # Stage 3
        # Report selection accuracy
        if MaleYT1.n_ind > 0 and FemaleYT1.n_ind > 0:
            male_cor = np.corrcoef(MaleYT1.pheno[:, 0], MaleYT1.gv[:, 0])[0, 1]
            female_cor = np.corrcoef(FemaleYT1.pheno[:, 0], FemaleYT1.gv[:, 0])[0, 1]
            output['acc_sel'][year - 1] = (male_cor + female_cor) / 2
        
        MaleYT2 = selectInd(MaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        FemaleYT2 = selectInd(FemaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        
        MaleYT2 = setPhenoGCA(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
        FemaleYT2 = setPhenoGCA(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
        
        # Stage 2
        MaleDH = makeDH(MaleF1, nDH=n_dh, simParam=SP)
        FemaleDH = makeDH(FemaleF1, nDH=n_dh, simParam=SP)
        
        MaleYT1 = setPhenoGCA(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
        FemaleYT1 = setPhenoGCA(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
        
        # Stage 1
        MaleF1 = randCross(MaleParents, nCrosses=n_crosses, simParam=SP)
        FemaleF1 = randCross(FemaleParents, nCrosses=n_crosses, simParam=SP)
        
        # Store training population
        if year == start_tp:
            print("  Start collecting training population")
            HybTrainPop = mergePops([MaleHybridYT3, MaleHybridYT4, MaleHybridYT5,
                                    FemaleHybridYT3, FemaleHybridYT4, FemaleHybridYT5])
            MaleTrainPop = mergePops([MaleInbredYT3, MaleInbredYT4, MaleInbredYT5])
            FemaleTrainPop = mergePops([FemaleInbredYT3, FemaleInbredYT4, FemaleInbredYT5])
        elif year > start_tp and year < n_burnin + 1:
            print("  Collecting training population")
            HybTrainPop = mergePops([HybTrainPop, MaleHybridYT3, MaleHybridYT4, MaleHybridYT5,
                                    FemaleHybridYT3, FemaleHybridYT4, FemaleHybridYT5])
            MaleTrainPop = mergePops([MaleTrainPop, MaleInbredYT3, MaleInbredYT4, MaleInbredYT5])
            FemaleTrainPop = mergePops([FemaleTrainPop, FemaleInbredYT3, FemaleInbredYT4, FemaleInbredYT5])
        
        # Report results
        output['meanG_inbred'][year - 1] = (meanG(MaleInbredYT3)[0] + meanG(FemaleInbredYT3)[0]) / 2
        output['varG_inbred'][year - 1] = (varG(MaleInbredYT3)[0] + varG(FemaleInbredYT3)[0]) / 2
        
        tmp_hybrid = hybridCross(FemaleInbredYT3, MaleInbredYT3, returnHybridPop=True, simParam=SP)
        output['meanG_hybrid'][year - 1] = meanG(tmp_hybrid)[0]
        output['varG_hybrid'][year - 1] = varG(tmp_hybrid)[0]
        
        tmp_gca = calcGCA(tmp_hybrid, use="gv")
        inbred_gv = np.concatenate([FemaleInbredYT3.gv[:, 0], MaleInbredYT3.gv[:, 0]])
        gca_values = np.concatenate([tmp_gca['GCAf'][:, 1], tmp_gca['GCAm'][:, 1]])
        output['cor'][year - 1] = np.corrcoef(inbred_gv, gca_values)[0, 1]
    
    # Future phase: Two-Part Genomic selection program
    print("\n--> Working on Two-Part Genomic hybrid program")
    
    # Initialize counter for PI cycles
    count = 0
    
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f" Working on future year: {year}")
        
        # Run genomic model
        print("  Running GS model")
        # Option 2: Hybrid additive GCA model (as in R reference)
        gsModel = RRBLUP(HybTrainPop, SP, traits=1, use="pheno", snpChip=1)
        
        # Update testers (pick new testers)
        best_male_idx = np.argmax(MaleHybridYT5.pheno[:, 0])
        best_male_inbred_id = MaleHybridYT5.mother[best_male_idx]
        best_male_inbred_idx = MaleInbredYT5.id.index(best_male_inbred_id)
        MaleElite_new = selectInd(MaleInbredYT5, nInd=1, parents=[best_male_inbred_idx], simParam=SP)
        MaleElite_old = selectInd(MaleElite, nInd=n_elite - 1, parents=list(range(1, n_elite)), simParam=SP)
        MaleElite = mergePops([MaleElite_old, MaleElite_new])
        
        best_female_idx = np.argmax(FemaleHybridYT5.pheno[:, 0])
        best_female_inbred_id = FemaleHybridYT5.mother[best_female_idx]
        best_female_inbred_idx = FemaleInbredYT5.id.index(best_female_inbred_id)
        FemaleElite_new = selectInd(FemaleInbredYT5, nInd=1, parents=[best_female_inbred_idx], simParam=SP)
        FemaleElite_old = selectInd(FemaleElite, nInd=n_elite - 1, parents=list(range(1, n_elite)), simParam=SP)
        FemaleElite = mergePops([FemaleElite_old, FemaleElite_new])
        
        MaleTester1 = selectInd(MaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
        FemaleTester1 = selectInd(FemaleElite, nInd=n_tester1, parents=list(range(n_tester1)), simParam=SP)
        MaleTester2 = selectInd(MaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)
        FemaleTester2 = selectInd(FemaleElite, nInd=n_tester2, parents=list(range(n_tester2)), simParam=SP)
        
        # Advance year (advances yield trials by a year and cycle parents)
        p = P[year - 1]
        
        # Product development
        print("   Product development")
        
        # Stage 6
        MaleHybridYT5 = selectInd(MaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        FemaleHybridYT5 = selectInd(FemaleHybridYT4, nInd=n_yt5, use="pheno", simParam=SP)
        
        MaleHybridYT5 = setPheno(MaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        FemaleHybridYT5 = setPheno(FemaleHybridYT5, varE=var_e, reps=rep_yt5, simParam=SP)
        
        MaleInbredYT5_ids = list(set(MaleHybridYT5.mother))
        FemaleInbredYT5_ids = list(set(FemaleHybridYT5.mother))
        MaleInbredYT5 = selectInd(MaleInbredYT4, nInd=len(MaleInbredYT5_ids),
                                parents=[MaleInbredYT4.id.index(id) for id in MaleInbredYT5_ids if id in MaleInbredYT4.id], simParam=SP)
        FemaleInbredYT5 = selectInd(FemaleInbredYT4, nInd=len(FemaleInbredYT5_ids),
                                   parents=[FemaleInbredYT4.id.index(id) for id in FemaleInbredYT5_ids if id in FemaleInbredYT4.id], simParam=SP)
        
        # Stage 5
        MaleHybridYT4 = selectInd(MaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        FemaleHybridYT4 = selectInd(FemaleHybridYT3, nInd=n_yt4, use="pheno", simParam=SP)
        
        MaleHybridYT4 = setPheno(MaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        FemaleHybridYT4 = setPheno(FemaleHybridYT4, varE=var_e, reps=rep_yt4, simParam=SP)
        
        MaleInbredYT4_ids = list(set(MaleHybridYT4.mother))
        FemaleInbredYT4_ids = list(set(FemaleHybridYT4.mother))
        MaleInbredYT4 = selectInd(MaleInbredYT3, nInd=len(MaleInbredYT4_ids),
                                 parents=[MaleInbredYT3.id.index(id) for id in MaleInbredYT4_ids if id in MaleInbredYT3.id], simParam=SP)
        FemaleInbredYT4 = selectInd(FemaleInbredYT3, nInd=len(FemaleInbredYT4_ids),
                                   parents=[FemaleInbredYT3.id.index(id) for id in FemaleInbredYT4_ids if id in FemaleInbredYT3.id], simParam=SP)
        
        # Stage 4
        MaleInbredYT3 = selectInd(MaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        FemaleInbredYT3 = selectInd(FemaleYT2, nInd=n_inbred3, use="pheno", simParam=SP)
        
        MaleHybridYT3 = hybridCross(MaleInbredYT3, FemaleElite, returnHybridPop=True, simParam=SP)
        FemaleHybridYT3 = hybridCross(FemaleInbredYT3, MaleElite, returnHybridPop=True, simParam=SP)
        
        MaleHybridYT3 = setPheno(MaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
        FemaleHybridYT3 = setPheno(FemaleHybridYT3, varE=var_e, reps=rep_yt3, simParam=SP)
        
        # Stage 3
        MaleYT2 = selectInd(MaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        FemaleYT2 = selectInd(FemaleYT1, nInd=n_inbred2, use="pheno", simParam=SP)
        
        MaleYT2 = setPhenoGCA(MaleYT2, FemaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
        FemaleYT2 = setPhenoGCA(FemaleYT2, MaleTester2, reps=rep_yt2, inbred=True, simParam=SP)
        
        # Stage 2
        MaleDH = makeDH(MaleF1, nDH=n_dh, simParam=SP)
        FemaleDH = makeDH(FemaleF1, nDH=n_dh, simParam=SP)
        
        # Apply genomic selection - predict GCA of DHs
        MaleDH = setEBV(MaleDH, gsModel, SP, snpChip=1)
        FemaleDH = setEBV(FemaleDH, gsModel, SP, snpChip=1)
        
        # Report average prediction accuracy across two pools
        if MaleDH.n_ind > 0 and FemaleDH.n_ind > 0:
            male_cor = np.corrcoef(MaleDH.ebv[:, 0], MaleDH.gv[:, 0])[0, 1]
            female_cor = np.corrcoef(FemaleDH.ebv[:, 0], FemaleDH.gv[:, 0])[0, 1]
            output['acc_sel'][year - 1] = (male_cor + female_cor) / 2
        
        # Make selection on EBVs
        MaleDH = selectWithinFam(MaleDH, nInd=fam_max, use="ebv", simParam=SP)
        FemaleDH = selectWithinFam(FemaleDH, nInd=fam_max, use="ebv", simParam=SP)
        MaleDH = selectInd(MaleDH, nInd=n_inbred2, use="ebv", simParam=SP)
        FemaleDH = selectInd(FemaleDH, nInd=n_inbred2, use="ebv", simParam=SP)
        
        # Grow testcross trials
        MaleYT1 = setPhenoGCA(MaleDH, FemaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
        FemaleYT1 = setPhenoGCA(FemaleDH, MaleTester1, reps=rep_yt1, inbred=True, simParam=SP)
        
        # Stage 1: Run population improvement
        print("   Population improvement")
        for cycle in range(1, n_cycles_pi + 1):
            print(f"    Population improvement cycle {cycle} / {n_cycles_pi}")
            
            if cycle == 1:
                count += 1
                
                if year == (n_burnin + 1):
                    # Create F1s by crossing parents from Burn-in
                    MaleParents = randCross(MaleParents, nCrosses=n_crosses, simParam=SP)
                    FemaleParents = randCross(FemaleParents, nCrosses=n_crosses, simParam=SP)
                
                # 1. Select best F1s using GS
                # Predict EBVs
                MaleParents = setEBV(MaleParents, gsModel, SP, snpChip=1)
                FemaleParents = setEBV(FemaleParents, gsModel, SP, snpChip=1)
                
                # Report average prediction accuracy across two pools
                if MaleParents.n_ind > 0 and FemaleParents.n_ind > 0:
                    male_cor = np.corrcoef(MaleParents.ebv[:, 0], MaleParents.gv[:, 0])[0, 1]
                    female_cor = np.corrcoef(FemaleParents.ebv[:, 0], FemaleParents.gv[:, 0])[0, 1]
                    acc_pi['accPI'][count - 1] = (male_cor + female_cor) / 2
                
                # F1s to advance to product development
                MaleF1 = selectInd(selectWithinFam(MaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                nInd=n_male_f1_pi, use="ebv", simParam=SP)
                FemaleF1 = selectInd(selectWithinFam(FemaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                  nInd=n_female_f1_pi, use="ebv", simParam=SP)
                
                # F1s to advance to next cycle as new parents
                MaleParents = selectInd(selectWithinFam(MaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                      nInd=n_parents_pi, use="ebv", simParam=SP)
                FemaleParents = selectInd(selectWithinFam(FemaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                        nInd=n_parents_pi, use="ebv", simParam=SP)
                
                # 2. Make parental crosses
                MaleParents = randCross(MaleParents, nCrosses=n_cross_male_pi, nProgeny=n_progeny_pi, simParam=SP)
                FemaleParents = randCross(FemaleParents, nCrosses=n_cross_female_pi, nProgeny=n_progeny_pi, simParam=SP)
            else:
                count += 1
                
                # 1. Select best F1s using GS
                # Predict EBVs
                MaleParents = setEBV(MaleParents, gsModel, SP, snpChip=1)
                FemaleParents = setEBV(FemaleParents, gsModel, SP, snpChip=1)
                
                # Report average prediction accuracy across two pools
                if MaleParents.n_ind > 0 and FemaleParents.n_ind > 0:
                    male_cor = np.corrcoef(MaleParents.ebv[:, 0], MaleParents.gv[:, 0])[0, 1]
                    female_cor = np.corrcoef(FemaleParents.ebv[:, 0], FemaleParents.gv[:, 0])[0, 1]
                    acc_pi['accPI'][count - 1] = (male_cor + female_cor) / 2
                
                # F1s to advance to next cycle as new parents
                MaleParents = selectInd(selectWithinFam(MaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                      nInd=n_parents_pi, use="ebv", simParam=SP)
                FemaleParents = selectInd(selectWithinFam(FemaleParents, nInd=max_fam_pi, use="ebv", simParam=SP), 
                                        nInd=n_parents_pi, use="ebv", simParam=SP)
                
                # 2. Make parental crosses
                MaleParents = randCross(MaleParents, nCrosses=n_cross_male_pi, nProgeny=n_progeny_pi, simParam=SP)
                FemaleParents = randCross(FemaleParents, nCrosses=n_cross_female_pi, nProgeny=n_progeny_pi, simParam=SP)
        
        # Store training population (maintain by removing oldest, adding newest)
        print("  Maintaining training population")
        # Remove oldest entries (size of hybrid YT3 + YT4 + YT5)
        n_remove = (MaleHybridYT3.n_ind + MaleHybridYT4.n_ind + MaleHybridYT5.n_ind +
                   FemaleHybridYT3.n_ind + FemaleHybridYT4.n_ind + FemaleHybridYT5.n_ind)
        if HybTrainPop.n_ind > n_remove:
            HybTrainPop = mergePops([subsetPop(HybTrainPop, list(range(n_remove, HybTrainPop.n_ind))),
                                    MaleHybridYT3, MaleHybridYT4, MaleHybridYT5,
                                    FemaleHybridYT3, FemaleHybridYT4, FemaleHybridYT5])
        else:
            HybTrainPop = mergePops([MaleHybridYT3, MaleHybridYT4, MaleHybridYT5,
                                    FemaleHybridYT3, FemaleHybridYT4, FemaleHybridYT5])
        
        # Report results
        output['meanG_inbred'][year - 1] = (meanG(MaleInbredYT3)[0] + meanG(FemaleInbredYT3)[0]) / 2
        output['varG_inbred'][year - 1] = (varG(MaleInbredYT3)[0] + varG(FemaleInbredYT3)[0]) / 2
        
        tmp_hybrid = hybridCross(FemaleInbredYT3, MaleInbredYT3, returnHybridPop=True, simParam=SP)
        output['meanG_hybrid'][year - 1] = meanG(tmp_hybrid)[0]
        output['varG_hybrid'][year - 1] = varG(tmp_hybrid)[0]
        
        tmp_gca = calcGCA(tmp_hybrid, use="gv")
        inbred_gv = np.concatenate([FemaleInbredYT3.gv[:, 0], MaleInbredYT3.gv[:, 0]])
        gca_values = np.concatenate([tmp_gca['GCAf'][:, 1], tmp_gca['GCAm'][:, 1]])
        output['cor'][year - 1] = np.corrcoef(inbred_gv, gca_values)[0, 1]
    
    # Save results from current replicate
    results.append(output)
    results_acc_pi.append(acc_pi)

print("\n" + "="*60)
print("Simulation completed!")
print("="*60)

## Analyze Results

Visualize the results from the simulation, including genetic gain, genetic variance, selection accuracy,
and selection accuracy in population improvement cycles.

In [ ]:
# Combine results from all replicates
df = pd.DataFrame(results[0])  # For single replicate, convert dict to DataFrame

# If multiple replicates, combine them
if len(results) > 1:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

print("Results summary:")
print(df.head(10))
print(f"\nTotal years simulated: {len(df)}")

In [ ]:
# Extract PI accuracy data
acc_pi_all = [r['accPI'] for r in results_acc_pi]
acc_pi_avg = np.mean(acc_pi_all, axis=0) if len(acc_pi_all) > 0 else []

# Plotting function
def plot_results(x, y, title, xlabel, ylabel, ylim=None):
    plt.plot(x, y, 'b-', linewidth=2)
    plt.axvline(x=n_burnin, color='r', linestyle='--', label='GS Start')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(ylim)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

# Create plots
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# Inbred Genetic Gain
plt.sca(axes[0, 0])
plot_results(df['year'], df['meanG_inbred'], 
              'Inbred genetic gain', 'Year', 'Yield')

# Hybrid Genetic Gain
plt.sca(axes[0, 1])
plot_results(df['year'], df['meanG_hybrid'], 
              'Hybrid genetic gain', 'Year', 'Yield')

# Inbred Variance
plt.sca(axes[1, 0])
plot_results(df['year'], df['varG_inbred'], 
              'Inbred genetic variance', 'Year', 'Variance')

# Hybrid Variance
plt.sca(axes[1, 1])
plot_results(df['year'], df['varG_hybrid'], 
              'Hybrid genetic variance', 'Year', 'Variance')

# Selection Accuracy in Product Development
plt.sca(axes[2, 0])
plot_results(df['year'], df['acc_sel'], 
              'Selection accuracy in Product Development', 'Year', 'Accuracy')

# Selection Accuracy in Population Improvement
plt.sca(axes[2, 1])
if len(acc_pi_avg) > 0:
    cycles_pi = np.arange(1, len(acc_pi_avg) + 1)
    plt.plot(cycles_pi, acc_pi_avg, 'b-', linewidth=2)
    plt.title('Selection accuracy in Population Improvement')
    plt.xlabel('Cycle')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='--', alpha=0.7)
else:
    plt.text(0.5, 0.5, 'No PI accuracy data', ha='center', va='center')
    plt.title('Selection accuracy in Population Improvement')

plt.tight_layout()
plt.savefig('TwoPartGS_Results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results plot saved as 'TwoPartGS_Results.png'")

## Summary

This tutorial demonstrated:

1. **Founder Population Creation**: Using `runMacs` with heterotic pool split to generate initial haplotypes
2. **Trait Definition**: Adding traits with additive, dominance, and GxE effects using `addTraitADG`
3. **Hybrid Breeding Pipeline**: Setting up a 6-stage hybrid breeding pipeline with:
   - F1 crosses
   - Doubled haploid (DH) line production
   - GCA evaluation using testers (YT1, YT2)
   - Hybrid evaluation (YT3, YT4, YT5)
4. **Two-Part Genomic Selection Strategy**:
   - **Product Development (PD)**: Conventional breeding pipeline with GS applied to advance DH lines to YT1
   - **Population Improvement (PI)**: Rapid cycling of parents with multiple cycles per year using GS
   - Training on hybrid phenotypes from YT3, YT4, and YT5
   - Predicting EBVs for DH lines and F1 parents
   - Selecting within families and across families based on EBVs
5. **Parent and Tester Updates**: Replacing parents and testers based on performance
6. **Training Population Management**: Maintaining a rolling training population by removing oldest entries
7. **Genetic Progress**: Tracking inbred and hybrid genetic gain, variance, selection accuracy in PD and PI, and correlations over time

The two-part strategy allows for rapid genetic gain through population improvement cycles while maintaining a conventional product development pipeline. Genomic selection accelerates both components by enabling early selection based on predicted breeding values.